# Частина 2: Аналіз споживання електроенергії
### Завдання 1: Завантаження та очистка даних
* Звантажити датасет "Individual Household Electric Power Consumption".
* Здійснити data cleaning (видалити пропуски, правильне форматування типів даних).

In [1]:
import pandas as pd
import urllib.request
import zipfile
import os

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00235/household_power_consumption.zip"
zip_path = "household_power_consumption.zip"
txt_path = "household_power_consumption.txt"

if not os.path.exists(txt_path):
    print("Завантажуємо архів (близько 20 МБ)...")
    urllib.request.urlretrieve(url, zip_path)
    print("Розпаковуємо...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall()
    print("Готово\n")

print("Зчитуємо дані у Pandas...")

df = pd.read_csv(txt_path, sep=';', na_values=['?'], low_memory=False)

df_cleaned = df.dropna().copy()

df_cleaned['Date'] = pd.to_datetime(df_cleaned['Date'], format='%d/%m/%Y')
df_cleaned['Time'] = pd.to_datetime(df_cleaned['Time'], format='%H:%M:%S').dt.time

print("Очистку завершено")
print(f"Кількість рядків після очистки: {len(df_cleaned)}")

df_cleaned.head()

Завантажуємо архів (близько 20 МБ)...
Розпаковуємо...
Готово

Зчитуємо дані у Pandas...
Очистку завершено
Кількість рядків після очистки: 2049280


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,2006-12-16,17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
1,2006-12-16,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,2006-12-16,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,2006-12-16,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
4,2006-12-16,17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


### Завдання 2.1: Вибірка за активною потужністю (> 5 кВт)
Обрати всі записи, у яких загальна активна споживана потужність (Global_active_power) перевищує 5 кВт. Профілювання часу виконання здійснюється за допомогою `timeit`.

In [2]:
import timeit

def filter_high_active_power(df):
    """Повертає датафрейм, де загальна активна споживана потужність > 5 кВт."""
    return df[df['Global_active_power'] > 5.0]

start_time = timeit.default_timer()

high_power_df = filter_high_active_power(df_cleaned)

execution_time = timeit.default_timer() - start_time

print("--- Результати Завдання 2.1 ---")
print(f"Знайдено записів: {len(high_power_df)}")
print(f"Час виконання фільтрації: {execution_time:.5f} секунд")

high_power_df.head()

--- Результати Завдання 2.1 ---
Знайдено записів: 17547
Час виконання фільтрації: 0.01517 секунд


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
1,2006-12-16,17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2,2006-12-16,17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
3,2006-12-16,17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
11,2006-12-16,17:35:00,5.412,0.470,232.78,23.2,0.0,1.0,17.0
12,2006-12-16,17:36:00,5.224,0.478,232.99,22.4,0.0,1.0,16.0


### Завдання 2.2: Вибірка за силою струму та споживанням приладів
Обрати всі записи, у яких:
1. Сила струму (Global_intensity) лежить в межах 19-20 А.
2. Пральна машина та холодильник (Sub_metering_2) споживають більше, ніж бойлер та кондиціонер (Sub_metering_3).

In [3]:
def filter_appliances(df):
    """Повертає датафрейм за умовами: струм 19-20 А, та пралка/холодильник > бойлер/кондиціонер."""
    return df[
        (df['Global_intensity'] >= 19.0) & 
        (df['Global_intensity'] <= 20.0) & 
        (df['Sub_metering_2'] > df['Sub_metering_3'])
    ]

start_time = timeit.default_timer()

appliances_df = filter_appliances(df_cleaned)

execution_time = timeit.default_timer() - start_time

print("--- Результати Завдання 2.2 ---")
print(f"Знайдено записів: {len(appliances_df)}")
print(f"Час виконання фільтрації: {execution_time:.5f} секунд")

appliances_df.head()

--- Результати Завдання 2.2 ---
Знайдено записів: 2509
Час виконання фільтрації: 0.02957 секунд


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
45,2006-12-16,18:09:00,4.464,0.136,234.66,19.0,0.0,37.0,16.0
460,2006-12-17,01:04:00,4.582,0.258,238.08,19.6,0.0,13.0,0.0
464,2006-12-17,01:08:00,4.618,0.104,239.61,19.6,0.0,27.0,0.0
475,2006-12-17,01:19:00,4.636,0.140,237.37,19.4,0.0,36.0,0.0
476,2006-12-17,01:20:00,4.634,0.152,237.17,19.4,0.0,35.0,0.0


### Завдання 2.3: Випадкова вибірка та середнє споживання
Обрати випадковим чином 500 000 записів (без повторів елементів вибірки), для них обчислити середні величини усіх 3-х груп споживання електричної енергії (Sub_metering_1, Sub_metering_2, Sub_metering_3).

In [4]:
import timeit

def sample_and_calculate_means(df):
    """Повертає середні значення 3-х груп для випадкових 500 000 записів."""
    sampled_df = df.sample(n=500000, replace=False, random_state=42)
    
    means = sampled_df[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].mean()
    
    return means, sampled_df

start_time = timeit.default_timer()

means_result, sampled_df = sample_and_calculate_means(df_cleaned)

execution_time = timeit.default_timer() - start_time

print("--- Результати Завдання 2.3 ---")
print(f"Час виконання: {execution_time:.5f} секунд")
print("\nСередні величини споживання (у ват-годинах активної енергії):")
print(means_result.round(3))

--- Результати Завдання 2.3 ---
Час виконання: 0.89193 секунд

Середні величини споживання (у ват-годинах активної енергії):
Sub_metering_1    1.119
Sub_metering_2    1.309
Sub_metering_3    6.453
dtype: float64


### Завдання 2.4: Складний фільтр та вибірка по половинах
* Обрати записи після 18:00 із загальним споживанням > 6 кВт.
* Визначити ті, де основне споживання припадає на 2-гу групу (пралка, холодильник тощо).
* Розділити результат навпіл: обрати кожен 3-й результат з першої половини та кожен 4-й з другої.

In [5]:
import timeit
import datetime

def complex_slicing_filter(df):
    
    time_limit = datetime.time(18, 0, 0)
    filtered = df[(df['Time'] >= time_limit) & (df['Global_active_power'] > 6.0)]
    
    filtered = filtered[
        (filtered['Sub_metering_2'] > filtered['Sub_metering_1']) & 
        (filtered['Sub_metering_2'] > filtered['Sub_metering_3'])
    ]
    
    half_index = len(filtered) // 2
    first_half = filtered.iloc[:half_index]
    second_half = filtered.iloc[half_index:]
    
    first_sampled = first_half.iloc[::3]
    second_sampled = second_half.iloc[::4]
    
    result = pd.concat([first_sampled, second_sampled])
    
    return result

start_time = timeit.default_timer()

complex_df = complex_slicing_filter(df_cleaned)

execution_time = timeit.default_timer() - start_time

print("--- Результати завдання 2.4 ---")
print(f"Знайдено записів після всіх фільтрів: {len(complex_df)}")
print(f"Час виконання: {execution_time:.5f} секунд")

complex_df.head()

--- Результати завдання 2.4 ---
Знайдено записів після всіх фільтрів: 310
Час виконання: 0.43679 секунд


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
41,2006-12-16,18:05:00,6.052,0.192,232.93,26.2,0.0,37.0,17.0
44,2006-12-16,18:08:00,6.308,0.116,232.25,27.0,0.0,36.0,17.0
17494,2006-12-28,20:58:00,6.386,0.374,236.63,27.0,1.0,36.0,17.0
17498,2006-12-28,21:02:00,8.088,0.262,235.50,34.4,1.0,72.0,17.0
17501,2006-12-28,21:05:00,7.230,0.152,235.22,30.6,1.0,73.0,17.0


### Завдання 3: Статистичний аналіз та підготовка даних
* Розрахувати кореляцію Пірсона та Спірмена для числових ознак.
* Здійснити стандартизацію даних (Z-score).
* Створити нову категоріальну змінну (день тижня) та виконати One-Hot Encoding.

In [6]:
import pandas as pd

print("--- 1. Кореляція пірсона та Спірмена ---")
num_cols = df_cleaned.select_dtypes(include=['float64'])

print("Кореляція Пірсона (лінійна залежність):")
display(num_cols.corr(method='pearson').round(2))

print("Кореляція Спірмена (рангова залежність):")
display(num_cols.corr(method='spearman').round(2))


print("\n--- 2. Стандартизація даних (Z-score) ---")
standardized_df = (num_cols - num_cols.mean()) / num_cols.std()

print("Перші 5 рядків стандартизованих даних:")
display(standardized_df.head().round(3))


print("\n--- 3. One-Hot Encoding ---")
df_sample = df_cleaned.head(10).copy()
df_sample['Day_of_Week'] = df_sample['Date'].dt.day_name()

encoded_days = pd.get_dummies(df_sample['Day_of_Week'], dtype=int)

result_ohe = pd.concat([df_sample[['Date', 'Day_of_Week']], encoded_days], axis=1)

print("Результат перетворення категорій у бінарні стовпці:")
display(result_ohe)

--- 1. Кореляція пірсона та Спірмена ---
Кореляція Пірсона (лінійна залежність):


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
Global_active_power,1.00,0.25,-0.40,1.00,0.48,0.43,0.64
Global_reactive_power,0.25,1.00,-0.11,0.27,0.12,0.14,0.09
Voltage,-0.40,-0.11,1.00,-0.41,-0.20,-0.17,-0.27
Global_intensity,1.00,0.27,-0.41,1.00,0.49,0.44,0.63
Sub_metering_1,0.48,0.12,-0.20,0.49,1.00,0.05,0.10
Sub_metering_2,0.43,0.14,-0.17,0.44,0.05,1.00,0.08
Sub_metering_3,0.64,0.09,-0.27,0.63,0.10,0.08,1.00


Кореляція Спірмена (рангова залежність):


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
Global_active_power,1.00,0.27,-0.33,1.00,0.34,0.19,0.60
Global_reactive_power,0.27,1.00,-0.09,0.30,0.12,0.43,0.07
Voltage,-0.33,-0.09,1.00,-0.35,-0.18,-0.09,-0.18
Global_intensity,1.00,0.30,-0.35,1.00,0.34,0.20,0.60
Sub_metering_1,0.34,0.12,-0.18,0.34,1.00,0.06,0.15
Sub_metering_2,0.19,0.43,-0.09,0.20,0.06,1.00,0.04
Sub_metering_3,0.60,0.07,-0.18,0.60,0.15,0.04,1.00



--- 2. Стандартизація даних (Z-score) ---
Перші 5 рядків стандартизованих даних:


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,2.955,2.611,-1.852,3.099,-0.182,-0.051,1.249
1,4.037,2.770,-2.225,4.134,-0.182,-0.051,1.131
2,4.050,3.320,-2.330,4.134,-0.182,0.120,1.249
3,4.064,3.356,-2.191,4.134,-0.182,-0.051,1.249
4,2.435,3.587,-1.593,2.514,-0.182,-0.051,1.249



--- 3. One-Hot Encoding ---
Результат перетворення категорій у бінарні стовпці:


,Date,Day_of_Week,Saturday
0,2006-12-16,Saturday,1
1,2006-12-16,Saturday,1
2,2006-12-16,Saturday,1
3,2006-12-16,Saturday,1
4,2006-12-16,Saturday,1
5,2006-12-16,Saturday,1
6,2006-12-16,Saturday,1
7,2006-12-16,Saturday,1
8,2006-12-16,Saturday,1
9,2006-12-16,Saturday,1
